### Prepare environment

In [0]:
%run ../../environment/prepare_environment_llm

# Notebook 9.1 - Decoder-Only Transformer

This notebook trains a small decoder-only language model on the workshop materials, tracks the run in MLflow, registers the model, and then uses it for text generation.

Treat this model as a microscope for understanding LLM mechanics. It is much smaller than a production foundation model, but the core loop is the same: turn text into token ids, learn to predict the next token, and generate text by repeating that prediction.

## 1. Configuration

This section defines where artifacts will be logged and how large the model should be. The important model settings are deliberately visible instead of hidden in a config file:

- `context_length` is the maximum number of previous tokens the model can see at once. Real LLMs also have a context limit; they can accept different prompt lengths, but each request is padded, trimmed, or packed into a bounded window internally.
- `vocab_size` controls how many distinct tokens the model knows. Tokens outside that list become `<unk>`.
- `embedding_dim`, `num_heads`, `num_layers`, and `ffn_dim` control model capacity. Larger values can learn richer patterns, but they also train more slowly.

The reusable implementation lives in `workshop_decoder.py`.

In [0]:
import os
import random
import sys
import time
from pathlib import Path

import mlflow
import numpy as np
import pandas as pd
import tensorflow as tf
from mlflow.models import infer_signature

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


def find_repo_root(repo_root_widget: str) -> Path:
    """Find the repository folder that contains the workshop files.

    The model trains on markdown files and notebook markdown cells from this repository, so
    every later path depends on locating the repo root correctly. In an imported
    Databricks notebook, the current working directory can vary; the widget is a
    manual override, and the parent-directory scan is the default.
    """
    if repo_root_widget:
        return Path(repo_root_widget).expanduser().resolve()
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "9_genai_llms").exists():
            return candidate
    raise FileNotFoundError("Could not locate repository root. Set repo_root manually.")


current_user = spark.sql("select current_user()").first()[0]

dbutils.widgets.text("repo_root", "")
dbutils.widgets.text("catalog_name", "ai_ml_in_practice")
dbutils.widgets.text("schema_name", "genai_workshop")
dbutils.widgets.text("registered_model_name", "workshop_decoder_transformer")
dbutils.widgets.text(
    "experiment_path", f"/Users/{current_user}/9_genai_decoder_transformer"
)
dbutils.widgets.text("context_length", "64")
dbutils.widgets.text("vocab_size", "4000")
dbutils.widgets.text("embedding_dim", "160")
dbutils.widgets.text("num_heads", "4")
dbutils.widgets.text("num_layers", "4")
dbutils.widgets.text("ffn_dim", "320")
dbutils.widgets.text("dropout", "0.1")
dbutils.widgets.text("batch_size", "64")
dbutils.widgets.text("epochs", "100")
dbutils.widgets.text("learning_rate", "0.001")
dbutils.widgets.text("seed", "42")

repo_root = find_repo_root(dbutils.widgets.get("repo_root").strip())
module_src = repo_root / "9_genai_llms" / "src"
if str(module_src) not in sys.path:
    sys.path.insert(0, str(module_src))

from workshop_decoder import (
    TOKEN_PATTERN,
    build_examples,
    build_model,
    build_vocabulary,
    decode_token_ids,
    encode_prompt,
    generate_text,
    load_workshop_corpus,
    pad_or_trim_context,
    tokenize,
    tokens_to_ids,
)

catalog_name = dbutils.widgets.get("catalog_name")
schema_name = dbutils.widgets.get("schema_name")
registered_model_name = dbutils.widgets.get("registered_model_name")
experiment_path = dbutils.widgets.get("experiment_path")
context_length = int(dbutils.widgets.get("context_length"))
vocab_size = int(dbutils.widgets.get("vocab_size"))
embedding_dim = int(dbutils.widgets.get("embedding_dim"))
num_heads = int(dbutils.widgets.get("num_heads"))
num_layers = int(dbutils.widgets.get("num_layers"))
ffn_dim = int(dbutils.widgets.get("ffn_dim"))
dropout = float(dbutils.widgets.get("dropout"))
batch_size = int(dbutils.widgets.get("batch_size"))
epochs = int(dbutils.widgets.get("epochs"))
learning_rate = float(dbutils.widgets.get("learning_rate"))
seed = int(dbutils.widgets.get("seed"))

full_model_name = f"{catalog_name}.{schema_name}.{registered_model_name}"
model_config = {
    "model_class": "DecoderOnlyTransformer",
    "context_length": context_length,
    "vocab_size": vocab_size,
    "embedding_dim": embedding_dim,
    "num_heads": num_heads,
    "num_layers": num_layers,
    "ffn_dim": ffn_dim,
    "dropout": dropout,
    "token_pattern": TOKEN_PATTERN,
}

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(experiment_path)

random.seed(seed)
np.random.seed(seed)
tf.keras.utils.set_random_seed(seed)

print(
    {
        "repo_root": str(repo_root),
        "experiment_path": experiment_path,
        "model": full_model_name,
        **model_config,
    }
)

## 2. Load and Tokenize Workshop Text

A language model learns statistical patterns from text. Here the corpus comes from the repository itself: markdown files and notebook markdown cells are read as plain text. Code cells are excluded so the model learns explanatory workshop language instead of Python syntax, SDK calls, and notebook paths.

The model cannot consume text strings directly, so the text is converted in two steps:

1. `tokenize` splits text into words and punctuation marks. For example, `LLMs use RAG.` becomes tokens like `llms`, `use`, `rag`, `.`.
2. `tokens_to_ids` replaces every token with an integer from the vocabulary, because embedding layers are lookup tables indexed by numbers.

The first two vocabulary entries are special. `<pad>` fills empty positions when a prompt is shorter than the context window. `<unk>` stands for an unknown token that was too rare to keep in the compact vocabulary.

In [0]:
corpus, source_paths = load_workshop_corpus(repo_root)
tokens = tokenize(corpus)
vocab = build_vocabulary(tokens, vocab_size)
model_config["vocab_size"] = len(vocab)
token_ids = tokens_to_ids(tokens, vocab)

print(
    {
        "source_files": len(source_paths),
        "characters": len(corpus),
        "tokens": len(tokens),
        "vocab_size": len(vocab),
    }
)

pd.DataFrame({"token": vocab[:30], "token_id": range(min(30, len(vocab)))})

## 3. Build Next-Token Training Examples

This is the central training idea behind decoder-only language models: the model sees previous tokens and learns to predict the next one.

The function `build_examples` creates two arrays:

- `x` contains the input context windows.
- `y` contains the same windows shifted one token forward.

Example: for tokens `machine learning pipelines train models`, one training pair can look like this:

- input: `machine learning pipelines train`
- target: `learning pipelines train models`

At position 1 the model learns that `learning` follows `machine`. At position 2 it learns that `pipelines` follows `machine learning`. The causal attention mask enforces that rule inside the transformer: a token can use itself and earlier tokens, but not future tokens.

In [0]:
x_all, y_all = build_examples(token_ids, context_length)
shuffle_indices = np.random.permutation(len(x_all))
x_all = x_all[shuffle_indices]
y_all = y_all[shuffle_indices]

validation_size = max(1, int(len(x_all) * 0.15))
x_val, y_val = x_all[:validation_size], y_all[:validation_size]
x_train, y_train = x_all[validation_size:], y_all[validation_size:]

train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=min(len(x_train), 5000), seed=seed)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_val, y_val))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

pd.DataFrame(
    {
        "input_token": [
            decode_token_ids([token_id], vocab) for token_id in x_train[0][:12]
        ],
        "target_next_token": [
            decode_token_ids([token_id], vocab) for token_id in y_train[0][:12]
        ],
    }
)

## 4. Train, Track, and Register Decoder-Only Transformer

The model is small, but the parts mirror a GPT-style decoder:

- token embeddings turn token ids into dense vectors;
- position embeddings tell the model where each token sits in the context;
- causal self-attention lets each position combine information from earlier positions;
- feed-forward layers transform the representation at each position;
- the final output head produces one raw score, called a logit, for every token in the vocabulary.

During training, the loss compares those logits with the shifted target sequence from the previous step. The full training step runs inside one MLflow run: parameters are logged before training, per-epoch metrics are logged from the Keras history, final metrics are logged after training, and the trained Keras model is registered from the same run.

In [0]:
mlflow_client = mlflow.tracking.MlflowClient()

with mlflow.start_run(run_name="decoder_only_transformer") as run:
    run_id = run.info.run_id
    mlflow.log_params(
        {
            **model_config,
            "batch_size": batch_size,
            "epochs_requested": epochs,
            "learning_rate": learning_rate,
            "train_examples": len(x_train),
            "validation_examples": len(x_val),
            "source_files": len(source_paths),
            "seed": seed,
        }
    )

    model = build_model(model_config)
    model.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=learning_rate, weight_decay=1e-4
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    model(np.zeros((1, context_length), dtype=np.int32))
    model.summary()

    start_time = time.time()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=epochs,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=8, restore_best_weights=True
            )
        ],
        verbose=1,
    )
    train_seconds = time.time() - start_time

    for epoch, values in enumerate(
        zip(
            history.history["loss"],
            history.history["accuracy"],
            history.history["val_loss"],
            history.history["val_accuracy"],
            strict=True,
        )
    ):
        train_loss, train_accuracy, val_loss, val_accuracy = values
        mlflow.log_metric("train_loss", float(train_loss), step=epoch)
        mlflow.log_metric("train_accuracy", float(train_accuracy), step=epoch)
        mlflow.log_metric("val_loss", float(val_loss), step=epoch)
        mlflow.log_metric("val_accuracy", float(val_accuracy), step=epoch)

    final_metrics = {
        "train_loss_last": float(history.history["loss"][-1]),
        "train_accuracy_last": float(history.history["accuracy"][-1]),
        "val_loss_last": float(history.history["val_loss"][-1]),
        "val_accuracy_last": float(history.history["val_accuracy"][-1]),
        "epochs_completed": len(history.history["loss"]),
        "train_seconds": float(train_seconds),
    }
    mlflow.log_metrics(final_metrics)

    signature_input = x_val[: min(8, len(x_val))]
    signature_output = model.predict(signature_input, verbose=0)
    signature = infer_signature(signature_input, signature_output)

    model_info = mlflow.tensorflow.log_model(
        model=model,
        artifact_path="model",
        signature=signature,
        input_example=signature_input[:1],
        registered_model_name=full_model_name,
        code_paths=[str(module_src / "workshop_decoder.py")],
    )

    mlflow.set_tags(
        {
            "registered_model_name": full_model_name,
            "registered_model_uri": model_info.model_uri,
            "model_flavor": "tensorflow",
        }
    )

latest_versions = mlflow_client.search_model_versions(f"name='{full_model_name}'")
registered_version = max(
    int(version.version) for version in latest_versions if version.run_id == run_id
)

training_result = {
    "run_id": run_id,
    "registered_model_name": full_model_name,
    "registered_model_version": registered_version,
    "model_uri": model_info.model_uri,
    **final_metrics,
}
training_result

## 5. Generate Text

Generation is the training task turned into a loop:

1. Encode the prompt into token ids.
2. Keep the latest `context_length` ids.
3. Ask the model for logits at the last position.
4. Sample one next token.
5. Append that token and repeat.

`temperature` changes how random sampling is. Lower values make the model choose safer, more likely tokens; higher values make it more exploratory. `top_k` limits sampling to the strongest candidates so a very unlikely token does not suddenly derail the output.

In [0]:
registered_model = mlflow.tensorflow.load_model(f"models:/{full_model_name}/1")

In [0]:
prompts = [
    "retrieval augmented generation",
    "machine learning pipelines",
    "a decoder only transformer",
]

generate_text(
    registered_model,
    prompt="machine learning pipelines",
    vocab=vocab,
    context_length=context_length,
    steps=25,
)

## 6. Inspect Causal Self-Attention

Self-attention is the mechanism that lets a token representation use other tokens from the same context. In a decoder-only model it must be causal: when predicting the next token at a given position, the model is allowed to use past tokens, but not future tokens.

The table below averages attention across heads from the first decoder block:

- rows are query positions: the token currently collecting information;
- columns are key positions: tokens that can be attended to;
- values are attention weights after the causal mask.

You should see that later tokens can place weight on earlier tokens, while future positions are blocked. This is not a full interpretability method, but it makes the masking rule visible.

In [0]:
token_to_id = {token: idx for idx, token in enumerate(vocab)}
attention_prompt = "retrieval augmented generation uses context from documents"
attention_ids = pad_or_trim_context(
    encode_prompt(attention_prompt, vocab),
    context_length,
    token_to_id["<pad>"],
)
_, attention_scores = registered_model(
    np.array([attention_ids], dtype=np.int32),
    training=False,
    return_attention=True,
)

attention_matrix = attention_scores.numpy()[0].mean(axis=0)
attention_tokens = [decode_token_ids([token_id], vocab) for token_id in attention_ids]

df = pd.DataFrame(
    np.round(attention_matrix, 3),
    index=[f"query:{token}" for token in attention_tokens],
    columns=[f"key:{token}" for token in attention_tokens],
)

num_real_tokens = len(encode_prompt(attention_prompt, vocab))
df.iloc[-num_real_tokens:, -num_real_tokens:]